# Prepare the inputs used by the figures

This notebook joins the saved DYNAMAP coordinates to participant metadata, training histories, and proposal rankings. Ideological labels are attached after estimation. It does not train a model.

**Inputs:** the fitted models in `outputs/models`, the source metadata and proposal labels in `data/raw`, and the age decisions and proposal rankings in `data/processed`. These files are included. Notebooks 01, 02–04, and 06 regenerate them.

**Outputs:** the spatial, composition, and training files in `data/processed/figure_inputs`, plus a sample-count summary and the combined training histories in `outputs`.

**Run or explore:** start a fresh kernel and run the cells from top to bottom. No other notebook must be run first to use the bundled fits. If you refit a model or recalculate rankings, rerun this notebook before drawing the affected figures. Unknown ages remain eligible for the maps. Demographic panels later apply their own complete-case rules.


The map inputs target two-dimensional fits. `SNAPSHOT_EPOCHS` selects saved training snapshots. The included fits have 20 epochs, but the history table retains whatever epochs were recorded.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/raw").is_dir(), "Open from the project root or notebooks directory."

SNAPSHOT_EPOCHS = [0, 5, 10, 20]
RAW = ROOT / "data/raw"
INPUT = ROOT / "data/processed/figure_inputs"
for name in ["composition", "spatial", "training"]:
    (INPUT / name).mkdir(parents=True, exist_ok=True)
TABLES = ROOT / "outputs/tables"
DIAGNOSTICS = ROOT / "outputs/diagnostics"
TABLES.mkdir(parents=True, exist_ok=True)
DIAGNOSTICS.mkdir(parents=True, exist_ok=True)
MODEL = {
    "Chile": ROOT / "outputs/models/Chile_dim2_seed101",
    "France": ROOT / "outputs/models/france",
    "Brazil": ROOT / "outputs/models/brazil",
}
decisions = pd.read_csv(
    ROOT / "data/processed/participant_age_decisions.csv", dtype={"uuid": str}
)
assert decisions.include.isin([True, False]).all()
metadata, coordinate_frames, sample_rows = {}, {}, []


## Participant and proposal coordinates

The exported metadata contain only variables used by the figures. Repeated responses are retained at this stage so that each figure can apply its visible consistency checks without selecting a response arbitrarily.

In [2]:
for country in MODEL:
    key = "uuid" if country == "Chile" else "user_id"
    coords = pd.read_csv(MODEL[country] / "coordsU.csv", dtype={key: str})
    assert coords[key].is_unique and coords[key].notna().all()
    age = decisions[decisions.country.eq(country)].set_index("uuid")
    # A one-to-one coordinate file must contain only age-eligible people.
    # Missing-age eligibility was decided upstream, not inferred from metadata here.
    assert set(coords[key]).issubset(age.index)
    assert age.loc[coords[key], "include"].all()
    assert np.isfinite(coords[["z1", "z2"]]).all().all()
    coordinate_frames[country] = coords
    display(coords)
    coords.to_csv(INPUT / "spatial" / f"{country}_coordsU.csv", index=False)

    source = pd.read_csv(RAW / f"{country.lower()}_metadata.csv", dtype={key: str})
    fields = [key, "age", "sex", "politica"] + (["datetime"] if country == "Chile" else [])
    meta = source.loc[source[key].isin(coords[key]), fields].copy()
    metadata[country] = meta
    display(meta)
    meta.to_csv(INPUT / "spatial" / f"{country}_metadata.csv", index=False)
    composition = meta[[key, "age", "sex", "politica"]].rename(columns={key: "participant_id"})
    display(composition)
    composition.to_csv(INPUT / "composition" / f"{country}_sample_metadata.csv", index=False)

    proposals = pd.read_csv(MODEL[country] / "coordsP.csv").rename(columns={"id": "option_id"})
    assert proposals.option_id.is_unique
    assert np.isfinite(proposals[["z1", "z2"]]).all().all()
    display(proposals)
    proposals.to_csv(INPUT / "spatial" / f"{country}_coordsP.csv", index=False)
    sample_rows.append(
        {
            "country": country,
            "map_participants": len(coords),
            "proposals": len(proposals),
            "metadata_rows": len(meta),
            "participants_with_metadata": meta[key].nunique(),
            "retained_without_metadata": len(set(coords[key]) - set(meta[key])),
        }
    )

sample_counts = pd.DataFrame(sample_rows)
display(sample_counts)
sample_counts.to_csv(DIAGNOSTICS / "publication_sample_counts.csv", index=False)
sample_counts.to_latex(DIAGNOSTICS / "publication_sample_counts.tex", index=False)


,uuid,z1,z2
0,000057c7-295c-4a61-8f62-76aff62bf079,0.620236,0.677677
1,000059b8-10b9-4f74-9c01-80bd2c48f509,0.371590,0.204995
2,00032199-2183-48c0-8f0d-ac37dc8ece60,0.143278,0.733773
3,000629ba-44c0-47c1-b2b2-abe4f0ee808f,0.287662,0.412808
4,0006b923-77a6-40c1-95e0-f98b817bcab2,0.262506,0.210308
...,...,...,...
47569,fff8a6be-746b-4d24-9148-73c8c6accb69,0.218051,0.444988
47570,fffa9813-765b-46c6-87f6-47fa854e784f,0.736183,0.269224
47571,fffcdf0e-0c78-40f7-915a-d41d6650c99b,0.278650,0.946591
47572,fffe76d8-ea19-4360-8050-1aadfb982c54,0.296505,0.188945


,uuid,age,sex,politica,datetime
1,8bd42737-b140-4db8-af4c-59839922129a,27,Masculino,2,2019-10-24 11:54:19.158423-03
2,06656cd5-a184-4f21-b947-2b82f83625f6,27,Femenino,5,2019-10-24 11:55:47.273066-03
3,3c573e0f-8f24-4202-8602-b9de2a53da73,39,Femenino,5,2019-10-24 11:56:40.908787-03
4,7d2e034a-3270-4429-aeac-418c46bf0d7d,43,Masculino,6,2019-10-24 11:56:42.384385-03
5,c1b97bb6-03d2-4027-8d54-0dd64a927a6c,39,Femenino,3,2019-10-24 11:56:57.166247-03
...,...,...,...,...,...
55939,98cb6edf-285b-4b08-8b58-db5108c9c936,43,Masculino,4,2019-11-01 14:36:24.689943-03
70907,047e5a25-7d49-4ccf-b4fb-a44ec5f710f4,36,Masculino,7,2019-11-02 23:22:45.346901-03
83473,a1164b76-094e-4018-9be0-b2b88fa3758a,41,Femenino,5,2019-11-04 08:35:02.586998-03
85743,4a0bbcc5-2d37-4983-8580-22988cd61f33,31,Femenino,1,2019-11-05 10:26:20.152761-03


,participant_id,age,sex,politica
1,8bd42737-b140-4db8-af4c-59839922129a,27,Masculino,2
2,06656cd5-a184-4f21-b947-2b82f83625f6,27,Femenino,5
3,3c573e0f-8f24-4202-8602-b9de2a53da73,39,Femenino,5
4,7d2e034a-3270-4429-aeac-418c46bf0d7d,43,Masculino,6
5,c1b97bb6-03d2-4027-8d54-0dd64a927a6c,39,Femenino,3
...,...,...,...,...
55939,98cb6edf-285b-4b08-8b58-db5108c9c936,43,Masculino,4
70907,047e5a25-7d49-4ccf-b4fb-a44ec5f710f4,36,Masculino,7
83473,a1164b76-094e-4018-9be0-b2b88fa3758a,41,Femenino,5
85743,4a0bbcc5-2d37-4983-8580-22988cd61f33,31,Femenino,1


,option_id,z1,z2
0,1,0.663117,0.460457
1,2,0.977909,0.769293
2,3,0.331054,0.274275
3,4,0.545465,0.685188
4,5,0.739270,0.115391
...,...,...,...
85,86,0.381413,0.672768
86,87,0.370188,0.254616
87,88,0.350024,0.830478
88,89,0.293969,0.046756


,user_id,z1,z2
0,001b18de-e566-4b13-9cf3-33e22e32be2a,0.638296,2.742960e-01
1,002323cc-f7bd-4c3b-b3a7-d909a4b48556,0.539659,3.823855e-01
2,0062701e-ba6f-420c-851c-a54557cad396,0.460885,6.715837e-01
3,007fb7f8-afde-4779-8fd9-191083118401,0.758623,4.740771e-07
4,0084be50-21a7-47bd-a21c-e917bb93969c,0.468171,4.531662e-01
...,...,...,...
1438,ff53f691-1512-46d4-b81d-15e0ff28ca8d,0.709750,1.082236e-03
1439,ff6f2650-d569-46e0-8ff2-2530cb6cd57d,0.232620,5.176508e-01
1440,ff7ee3e1-a916-4f4c-bc12-f3c9fc293e4c,0.499971,4.392747e-01
1441,ff8b80f1-5ca1-4bf6-9dd5-f2c14047f507,0.347375,9.649134e-02


...

,user_id,z1,z2
0,004ddaed-61da-408f-9030-79728dd9acfa,0.614649,0.329158
1,008945db-7336-4b6f-80c8-67f6f760146a,0.603311,0.459762
2,00f4f1e4-90a3-41ee-a0dc-b0fe3a824688,0.497200,0.506703
3,01048f40-73fa-460a-8a0f-d37f46ba2bf0,0.540310,0.481194
4,015ea9da-e6d4-4902-990e-298155884106,0.365017,0.535492
...,...,...,...
760,fe7adca8-9f18-4048-bd9a-a0d1c7636378,0.418282,0.515607
761,fe8ee69a-b37b-4e1b-be58-9391d29f0727,0.289338,0.258464
762,fecc1142-5009-42bc-b2c8-389c77e7b1fd,0.498278,0.477491
763,fed0ccb0-fdf8-423a-abd1-cef80cc4d6c5,0.667909,0.484973


,user_id,age,sex,politica
0,d351d60b-5592-4d58-8632-aa34fe4b8f07,4,2,3
1,e3428c81-c1a3-4139-9deb-ab2b9aae481d,4,2,1
2,248860f8-a008-4207-84fd-a88cdc02e3f1,5,2,2
3,f3d2061a-9ffa-4fc2-bf46-f9990b6055ec,3,1,2
4,457cc0fc-957a-4984-a685-587b0686d18d,4,2,2
...,...,...,...,...
784,5d5f537f-a895-49ed-966d-a5b31539155d,5,1,5
785,035fef84-952f-40cf-b6bc-59419ebc4234,6,1,5
786,730666d4-bf22-4c25-9df0-cd447cd72100,6,2,5
787,464341cf-d0ec-4067-8905-a50b34a7fd90,4,2,5


,participant_id,age,sex,politica
0,d351d60b-5592-4d58-8632-aa34fe4b8f07,4,2,3
1,e3428c81-c1a3-4139-9deb-ab2b9aae481d,4,2,1
2,248860f8-a008-4207-84fd-a88cdc02e3f1,5,2,2
3,f3d2061a-9ffa-4fc2-bf46-f9990b6055ec,3,1,2
4,457cc0fc-957a-4984-a685-587b0686d18d,4,2,2
...,...,...,...,...
784,5d5f537f-a895-49ed-966d-a5b31539155d,5,1,5
785,035fef84-952f-40cf-b6bc-59419ebc4234,6,1,5
786,730666d4-bf22-4c25-9df0-cd447cd72100,6,2,5
787,464341cf-d0ec-4067-8905-a50b34a7fd90,4,2,5


,option_id,z1,z2
0,1,0.158468,0.732182
1,2,0.628181,0.719921
2,3,0.730366,0.265257
3,4,0.698903,0.409594
4,5,0.696382,0.377449
...,...,...,...
62,63,0.554026,0.335928
63,65,0.596552,0.266331
64,66,0.513765,0.764957
65,67,0.450624,0.592414


,country,map_participants,proposals,metadata_rows,participants_with_metadata,retained_without_metadata
0,Chile,47574,90,42471,37636,9938
1,France,1443,120,1182,1129,314
2,Brazil,765,67,696,649,116


## Proposal text labels

The source label file provides proposal names for annotation. It contains no fitted coordinates.


In [3]:
labels = pd.read_csv(RAW / "chile_labels.tsv", sep="\t")
display(labels)
labels.to_csv(INPUT / "spatial/chile_labels.tsv", sep="\t", index=False)


,id,short_name,name,topic_name
0,1,Farmacias,Creación de Farmacias Populares,Salud
1,2,Marihuana,Legalización de la Marihuana,Salud
2,3,Cárcel Cohecho,Cárcel Efectiva para Soborno y Cohecho,Política Criminal
3,4,Retiro AFP,Retiro Anticipado Fondo de Pensiones,Pensiones
4,5,Más Metro,Crear Nuevas Líneas de Metro,Vivienda y Transporte
...,...,...,...,...
215,216,Transporte marítimo estatal,"Asegurar el transporte marítimo, del sur de Ch...",Vivienda y Transporte
216,217,Financiamiento campañas políticas,Modificación de ley electoral y financiamiento...,Democracia
217,218,Creación de un diccionario constitucional,Creación de un Diccionario Constitucional virt...,Educación
218,219,Policías Municipales,"Creación de Policías Municipales, eliminando C...",Modernización del Estado


## Training trajectories

Figure 2b uses snapshots from the same main Chilean fit at epochs 0, 5, 10, and 20. Each participant appears once. Scores 0–1 and 9–10 identify the displayed groups only. Participants who reported conflicting exact scores are excluded from this illustration, without being removed from model estimation.

The snapshot join below is one-to-one so repeated questionnaire responses cannot increase a person's weight. Figure S1 uses the recorded histories of the three country fits. These are the original optimization histories, not a new training run.


In [4]:
labels = metadata["Chile"][["uuid", "politica"]].drop_duplicates()
labels = labels[labels.politica.isin(range(11))]
conflicting = labels.groupby("uuid").politica.nunique().gt(1)
labels = labels[~labels.uuid.isin(conflicting[conflicting].index)].copy()
assert labels.uuid.is_unique
for epoch in SNAPSHOT_EPOCHS:
    snapshot = pd.read_csv(MODEL["Chile"] / f"coordsU_epoch_{epoch}.csv", dtype={"uuid": str})
    assert set(snapshot.uuid) == set(coordinate_frames["Chile"].uuid)
    snapshot = snapshot.merge(labels, on="uuid", validate="one_to_one")
    assert snapshot.uuid.is_unique
    display(snapshot)
    snapshot.to_csv(INPUT / "training" / f"coordsU_epoch_{epoch}.csv", index=False)

histories = []
rename = {
    "epoch": "Epoch",
    "accuracy": "Training Accuracy",
    "val_accuracy": "Validation Accuracy",
    "loss": "Training Loss",
    "val_loss": "Validation Loss",
}
for country in MODEL:
    path = (
        MODEL[country] / "history.csv"
        if country == "Chile"
        else ROOT / "outputs/tables" / f"{country.lower()}_training_history.csv"
    )
    history = pd.read_csv(path).rename(columns=rename)
    history = history[list(rename.values())].copy()
    history.insert(1, "Country", country)
    assert history.Epoch.tolist() == list(range(1, len(history) + 1))
    histories.append(history)
training = pd.concat(histories, ignore_index=True)
display(training)
training.to_csv(INPUT / "training/DataTraining_and_Validation_Accuracy_Loss.csv", index=False)
training.to_latex(
    ROOT / "outputs/tables/main_training_histories.tex", index=False, float_format="%.6f"
)


,uuid,z1,z2,politica
0,000057c7-295c-4a61-8f62-76aff62bf079,0.489714,0.525364,8
1,000059b8-10b9-4f74-9c01-80bd2c48f509,0.509358,0.547380,5
2,00032199-2183-48c0-8f0d-ac37dc8ece60,0.502152,0.522752,6
3,000629ba-44c0-47c1-b2b2-abe4f0ee808f,0.522003,0.512179,3
4,0006b923-77a6-40c1-95e0-f98b817bcab2,0.517174,0.503557,3
...,...,...,...,...
37095,fff8a6be-746b-4d24-9148-73c8c6accb69,0.502448,0.503297,3
37096,fffa9813-765b-46c6-87f6-47fa854e784f,0.521067,0.514581,5
37097,fffcdf0e-0c78-40f7-915a-d41d6650c99b,0.487167,0.505814,2
37098,fffe76d8-ea19-4360-8050-1aadfb982c54,0.514680,0.493731,5


,uuid,z1,z2,politica
0,000057c7-295c-4a61-8f62-76aff62bf079,0.535814,0.522041,8
1,000059b8-10b9-4f74-9c01-80bd2c48f509,0.502766,0.317414,5
2,00032199-2183-48c0-8f0d-ac37dc8ece60,0.373948,0.650328,6
3,000629ba-44c0-47c1-b2b2-abe4f0ee808f,0.465467,0.462574,3
4,0006b923-77a6-40c1-95e0-f98b817bcab2,0.466338,0.417493,3
...,...,...,...,...
37095,fff8a6be-746b-4d24-9148-73c8c6accb69,0.387831,0.492948,3
37096,fffa9813-765b-46c6-87f6-47fa854e784f,0.643933,0.405762,5
37097,fffcdf0e-0c78-40f7-915a-d41d6650c99b,0.388221,0.715154,2
37098,fffe76d8-ea19-4360-8050-1aadfb982c54,0.463757,0.378924,5


,uuid,z1,z2,politica
0,000057c7-295c-4a61-8f62-76aff62bf079,0.562038,0.566933,8
1,000059b8-10b9-4f74-9c01-80bd2c48f509,0.442161,0.236545,5
2,00032199-2183-48c0-8f0d-ac37dc8ece60,0.265156,0.716342,6
3,000629ba-44c0-47c1-b2b2-abe4f0ee808f,0.414716,0.447418,3
4,0006b923-77a6-40c1-95e0-f98b817bcab2,0.386616,0.326018,3
...,...,...,...,...
37095,fff8a6be-746b-4d24-9148-73c8c6accb69,0.317256,0.502190,3
37096,fffa9813-765b-46c6-87f6-47fa854e784f,0.699393,0.334416,5
37097,fffcdf0e-0c78-40f7-915a-d41d6650c99b,0.355269,0.837917,2
37098,fffe76d8-ea19-4360-8050-1aadfb982c54,0.400937,0.270211,5


,uuid,z1,z2,politica
0,000057c7-295c-4a61-8f62-76aff62bf079,0.620236,0.677677,8
1,000059b8-10b9-4f74-9c01-80bd2c48f509,0.371590,0.204995,5
2,00032199-2183-48c0-8f0d-ac37dc8ece60,0.143278,0.733773,6
3,000629ba-44c0-47c1-b2b2-abe4f0ee808f,0.287662,0.412808,3
4,0006b923-77a6-40c1-95e0-f98b817bcab2,0.262506,0.210308,3
...,...,...,...,...
37095,fff8a6be-746b-4d24-9148-73c8c6accb69,0.218051,0.444988,3
37096,fffa9813-765b-46c6-87f6-47fa854e784f,0.736183,0.269224,5
37097,fffcdf0e-0c78-40f7-915a-d41d6650c99b,0.278650,0.946591,2
37098,fffe76d8-ea19-4360-8050-1aadfb982c54,0.296505,0.188945,5


,Epoch,Country,Training Accuracy,Validation Accuracy,Training Loss,Validation Loss
0,1,Chile,0.665072,0.664806,0.610236,0.611114
1,2,Chile,0.684641,0.680313,0.589495,0.597038
2,3,Chile,0.700570,0.686230,0.571627,0.589006
3,4,Chile,0.707414,0.689353,0.562798,0.584372
4,5,Chile,0.710255,0.689268,0.558853,0.584209
5,6,Chile,0.712011,0.689787,0.556674,0.583044
6,7,Chile,0.712960,0.689178,0.555557,0.586063
7,8,Chile,0.713375,0.688947,0.555012,0.585880
8,9,Chile,0.713529,0.687716,0.554830,0.590195
9,10,Chile,0.713610,0.687454,0.554812,0.588780


## Proposal rankings

Notebook 06 calculates the left- and right-group TrueSkill rankings from retained Chilean choices. The uniqueness and ID-set checks ensure that every proposal on the map has exactly one ranking row before these values are used to color proposals.


In [5]:
rankings = pd.read_csv(ROOT / "data/processed/proposal_rankings.csv")
assert rankings.id.is_unique
assert set(rankings.id) == set(pd.read_csv(MODEL["Chile"] / "coordsP.csv").id)
display(rankings)
rankings.to_csv(INPUT / "spatial/Ranking_propuestas_trueskill_primerciclo.csv", index=False)


,id,Ranking Izquierda,skill iz,rank iz,Ranking Derecha,skill der,rank der,short_name,topic_name,name,ranking_derecha_sort,ranking_izquierda_sort,ranking_derecha_sort_skill,ranking_izquierda_sort_skill,ranking_derecha_sort_rank,ranking_izquierda_sort_rank
0,23,1,50.592856,1.07,75,39.381483,75.58,Nueva Constitución,Modernización del Estado,Nueva Constitución para Chile,75.000000,0.013333,0.778400,1.284686,70.635514,0.014157
1,55,2,48.919671,2.61,3,47.148776,6.02,Pensión Mínima,Pensiones,Pensión Mínima Igual o Mayor al Sueldo Mínimo,1.500000,0.666667,0.963800,1.037560,2.306513,0.433555
2,6,3,48.724362,3.12,46,42.499481,45.96,Agua,Economía,Desprivatización del Agua,15.333333,0.065217,0.872243,1.146470,14.730769,0.067885
3,85,4,47.978246,4.87,30,44.085301,32.53,Progresividad de Impuestos,Política Tributaria,Aumentar progresividad de impuestos (más impue...,7.500000,0.133333,0.918860,1.088305,6.679671,0.149708
4,37,5,47.970210,5.03,88,36.664592,88.08,Asamblea Constituyente,Modernización del Estado,Asamblea Constituyente,17.600000,0.056818,0.764320,1.308352,17.510934,0.057107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,11,86,36.156062,85.80,81,38.742262,80.41,Educación RRSS,Educación,Educación sobre uso de Redes Sociales,0.941860,1.061728,1.071529,0.933246,0.937179,1.067031
86,80,87,35.666375,86.66,65,40.399510,66.50,Autopistas,Vivienda y Transporte,Más Autopistas Urbanas,0.747126,1.338462,1.132706,0.882842,0.767367,1.303158
87,49,88,34.507068,88.80,87,38.031289,84.18,Venta de Alcohol,Salud,"Restringir Venta de Alcohol en Delivery, Super...",0.988636,1.011494,1.102130,0.907334,0.947973,1.054882
88,52,89,34.503466,88.71,68,40.373771,66.73,Edad de Jubilación,Pensiones,Aumento Edad de Jubilación,0.764045,1.308824,1.170137,0.854601,0.752226,1.329387
